# 📚 MIDA507 — Big Data, Open Data et Data Science pour l'IDA
## Notebook Session 02 · Cycle de Vie des Données et Évaluation FAIR

---

| | |
|---|---|
| **Cours** | MIDA507 — Big Data, Open Data et Data Science pour l'IDA |
| **Programme** | Master MIDA — Université de Douala |
| **Session** | S02 — Cycle de Vie des Données et Rôle du Data Steward |
| **Outil** | Google Colaboratory (Python 3) |
| **Durée estimée** | 60 à 90 minutes |
| **Prérequis** | Avoir exécuté le notebook S01 (ou télécharger le CSV fourni) |

---

### 🎯 Ce que vous allez faire dans ce notebook

1. **Reconstituer le cycle de vie DCC** d'un jeu de données africain en 7 étapes
2. **Évaluer automatiquement les principes FAIR** (Findable, Accessible, Interoperable, Reusable) sur notre jeu
3. **Identifier les lacunes FAIR** et générer des recommandations actionables pour l'IDA
4. **Construire la structure d'un DMP** (Data Management Plan) en Python
5. **Documenter le data lineage** (traçabilité des transformations)

### 📦 Jeux de données utilisés

| Fichier | Origine | Rôle dans ce notebook |
|---|---|---|
| `MIDA507_S01_bibliotheque_ngaoundere_clean.csv` | Produit en S01 | Jeu de base à évaluer et enrichir |
| Jeu dégradé généré ici | Simulé | Illustrer les problèmes FAIR réels |
| Jeu de données INS Cameroun simulé | Simulé (réaliste) | Évaluation FAIR sur données statistiques africaines |

---

---
## PARTIE 1 · Configuration et Chargement des Données


In [ ]:
# ============================================================
# CELLULE 1.1 — Imports et configuration
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import json
import os
import hashlib
import random
import warnings
from datetime import datetime, timedelta
from collections import OrderedDict

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (13, 5)
sns.set_theme(style="whitegrid")

# Palette MIDA507 (terracotta)
PAL = {
    "ter1": "#3E1500", "ter2": "#BF360C", "ter3": "#E64A19",
    "ter4": "#FF7043", "olive": "#558B2F", "bleu": "#1565C0",
    "gris": "#546E7A", "clair": "#FBE9E7"
}

print("✅ Bibliothèques chargées.")
print(f"   pandas {pd.__version__} | numpy {np.__version__}")


In [ ]:
# ============================================================
# CELLULE 1.2 — Chargement du jeu de données S01
# ============================================================
# Option A : Si vous avez le fichier CSV de S01, uploadez-le
#            (cliquez sur l'icône 📁 dans le panneau gauche de Colab)
# Option B : On le régénère ici automatiquement (même graine aléatoire)

def generer_jeu_s01(n=5000):
    '''Régénère le jeu de données de la session 01.'''
    random.seed(42); np.random.seed(42)
    genres    = ["Thèse et mémoire","Manuel universitaire","Revue scientifique",
                 "Rapport de recherche","Ouvrage de référence",
                 "Document administratif","Atlas et cartographie","Périodique"]
    p_genres  = [0.28,0.30,0.15,0.10,0.07,0.05,0.03,0.02]
    filieres  = ["Droit","Sciences économiques","Lettres modernes","Histoire",
                 "Géographie","Médecine","Agronomie","Informatique",
                 "Sciences de l'éducation","Sociologie"]
    p_fil     = [0.18,0.15,0.12,0.10,0.08,0.12,0.08,0.09,0.05,0.03]
    niveaux   = ["Licence 1","Licence 2","Licence 3",
                 "Master 1","Master 2","Doctorat","Enseignant-chercheur"]
    p_niv     = [0.20,0.18,0.17,0.15,0.12,0.10,0.08]
    regions   = ["Adamaoua","Centre","Est","Extrême-Nord","Littoral",
                 "Nord","Nord-Ouest","Ouest","Sud","Sud-Ouest"]
    p_reg     = [0.22,0.18,0.06,0.12,0.14,0.10,0.05,0.07,0.03,0.03]
    langues   = ["Français","Anglais","Bilingue FR/EN","Arabe","Autres langues africaines"]
    p_lang    = [0.62,0.22,0.10,0.04,0.02]

    d0 = datetime(2018,1,1); d1 = datetime(2023,12,31)
    delta = (d1-d0).days
    dates = [d0 + timedelta(days=random.randint(0, delta)) for _ in range(n)]
    durees = np.random.choice([3,7,7,7,14,14,14,21,21,30,60], n,
                              p=[0.05,0.15,0.15,0.15,0.10,0.10,0.10,0.08,0.05,0.04,0.03])
    df = pd.DataFrame({
        "cote":            [f"UNG-{str(i+1).zfill(5)}" for i in range(n)],
        "genre_document":  np.random.choice(genres, n, p=p_genres),
        "date_emprunt":    [d.strftime("%Y-%m-%d") for d in dates],
        "duree_jours":     durees,
        "code_usager":     [f"USR{str(random.randint(1,800)).zfill(4)}" for _ in range(n)],
        "filiere":         np.random.choice(filieres, n, p=p_fil),
        "niveau":          np.random.choice(niveaux, n, p=p_niv),
        "region_origine":  np.random.choice(regions, n, p=p_reg),
        "langue_document": np.random.choice(langues, n, p=p_lang),
    })
    df["annee"] = pd.to_datetime(df["date_emprunt"]).dt.year
    idx_nan = np.random.choice(df.index, size=int(n*0.05), replace=False)
    df.loc[idx_nan, "filiere"] = np.nan
    df.loc[df["duree_jours"] > 90, "duree_jours"] = 90  # nettoyé en S01
    df["filiere"] = df["filiere"].fillna("Non renseignée")
    return df

# Tenter de charger le fichier S01, sinon le régénérer
NOM_CSV_S01 = "MIDA507_S01_bibliotheque_ngaoundere_clean.csv"
if os.path.exists(NOM_CSV_S01):
    df_base = pd.read_csv(NOM_CSV_S01)
    print(f"✅ Fichier S01 chargé depuis le disque : {len(df_base):,} lignes")
else:
    df_base = generer_jeu_s01()
    print(f"✅ Jeu S01 régénéré automatiquement   : {len(df_base):,} lignes")

print(f"   Colonnes : {list(df_base.columns)}")
df_base.head(3)


---
## PARTIE 2 · Le Cycle de Vie des Données — Modèle DCC en 7 Étapes

> **Rappel théorique :** Le Digital Curation Centre (DCC) a formalisé un cycle de vie des données en 7 étapes circulaires. L'IDA intervient activement à chaque étape. Dans cette partie, nous reconstituons ce cycle pour notre jeu de données et documentons les actions réalisées.


In [ ]:
# ============================================================
# CELLULE 2.1 — Schéma visuel du cycle de vie DCC
# ============================================================
# On dessine le cycle de vie circulaire avec les 7 étapes
# et on indique le rôle IDA à chaque étape.

fig, ax = plt.subplots(figsize=(12, 12))
ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
ax.axis("off")

# Cercle central
cercle_centre = plt.Circle((0, 0), 0.38, color=PAL["ter1"],
                             zorder=5, alpha=0.95)
ax.add_patch(cercle_centre)
ax.text(0, 0.07, "CYCLE DE VIE", ha="center", va="center",
        fontsize=11, fontweight="bold", color="white", zorder=6)
ax.text(0, -0.05, "DES DONNÉES", ha="center", va="center",
        fontsize=11, fontweight="bold", color="white", zorder=6)
ax.text(0, -0.20, "(DCC)", ha="center", va="center",
        fontsize=10, color="#FFAB91", zorder=6)

# Cercle de fond
cercle_fond = plt.Circle((0, 0), 1.2, color=PAL["clair"],
                           zorder=1, alpha=0.3)
ax.add_patch(cercle_fond)

# 7 étapes du cycle
etapes = [
    ("1. PLANIFICATION",  90,  PAL["ter1"],  "★★★ IDA
DMP, schéma
métadonnées"),
    ("2. COLLECTE",       141, PAL["ter2"],  "★★★ IDA
Provenance
droits, qualité"),
    ("3. TRAITEMENT",     192, PAL["ter3"],  "★★★ IDA
OpenRefine
traçabilité"),
    ("4. ANALYSE",        243, "#FF8A65",    "★ IDA
Contextualiser
biais"),
    ("5. PUBLICATION",    294, "#FF7043",    "★★★ IDA
DCAT, licence
open data"),
    ("6. PRÉSERVATION",   345, PAL["olive"], "★★★ IDA
Formats pérennes
checksum"),
    ("7. RÉUTILISATION",  36,  PAL["bleu"],  "★★ IDA
Droits, retours
méta à jour"),
]

for nom, angle_deg, couleur, role in etapes:
    angle_rad = np.radians(angle_deg)
    # Position de la boîte
    r_boite = 0.95
    x_b = r_boite * np.cos(angle_rad)
    y_b = r_boite * np.sin(angle_rad)

    # Boîte de l'étape
    boite = plt.FancyBboxPatch((x_b - 0.18, y_b - 0.10), 0.36, 0.20,
                                boxstyle="round,pad=0.02",
                                facecolor=couleur, edgecolor="white",
                                linewidth=1.5, zorder=4)
    ax.add_patch(boite)
    ax.text(x_b, y_b, nom, ha="center", va="center",
            fontsize=8.5, fontweight="bold", color="white", zorder=5)

    # Rôle IDA (plus petit, à l'extérieur)
    r_role = 1.32
    x_r = r_role * np.cos(angle_rad)
    y_r = r_role * np.sin(angle_rad)
    ax.text(x_r, y_r, role, ha="center", va="center",
            fontsize=7.5, color=PAL["ter1"],
            bbox=dict(boxstyle="round,pad=0.15", facecolor="white",
                      edgecolor=couleur, alpha=0.9))

    # Flèche du centre vers la boîte
    ax.annotate("", xy=(x_b * 0.55, y_b * 0.55),
                xytext=(x_b * 0.40, y_b * 0.40),
                arrowprops=dict(arrowstyle="->",
                                color=couleur, lw=1.5),
                zorder=3)

ax.set_title("Cycle de Vie des Données — Modèle DCC
avec Rôle du Data Steward IDA",
             fontsize=14, fontweight="bold", color=PAL["ter1"], pad=15)

# Légende
for label, couleur in [("★★★ Rôle central IDA", PAL["ter1"]),
                        ("★★  Rôle important", PAL["bleu"]),
                        ("★    Rôle secondaire", "#FF8A65")]:
    ax.add_patch(mpatches.Patch(facecolor=couleur, label=label))
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.02),
          ncol=3, fontsize=9, framealpha=0.9)

plt.tight_layout()
plt.show()

print("💡 L'IDA est l'acteur central de 5 étapes sur 7 du cycle de vie.")
print("   Dans ce notebook, nous allons parcourir chaque étape sur")
print("   notre jeu de données de bibliothèque universitaire camerounaise.")


In [ ]:
# ============================================================
# CELLULE 2.2 — Documenter le cycle de vie de notre jeu de données
# ============================================================
# On crée un registre du cycle de vie : pour chaque étape,
# on documente ce qui a été fait, par qui et avec quel outil.

cycle_vie_journal = OrderedDict({
    "identifiant_jeu":   "MIDA507-BIBLIO-UNG-2018-2023",
    "institution":       "Bibliothèque Centrale — Université de Ngaoundéré",
    "date_creation_journal": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "etapes": [
        {
            "numero": 1,
            "nom": "Planification",
            "statut": "✅ Réalisé",
            "date": "2018-01-01",
            "responsable": "Direction de la Bibliothèque + Data Steward IDA",
            "actions": [
                "Définition du schéma de saisie des emprunts (10 champs)",
                "Choix du format de stockage : CSV UTF-8",
                "Définition des règles de qualité minimales",
                "Planification de la durée de conservation : 10 ans"
            ],
            "outil": "Document Word — Politique de données v1.0",
            "lacunes_identifiees": "Aucun DMP formalisé — à corriger"
        },
        {
            "numero": 2,
            "nom": "Collecte",
            "statut": "✅ Réalisé",
            "date": "2018-01-01 / 2023-12-31",
            "responsable": "Agents de prêt (saisie) + IDA (supervision)",
            "actions": [
                "Saisie quotidienne dans le SIGB (PMB)",
                "Export CSV mensuel du SIGB",
                "Vérification hebdomadaire des saisies par le responsable IDA"
            ],
            "outil": "PMB (SIGB open source)",
            "lacunes_identifiees": "5% de champs 'filière' non renseignés"
        },
        {
            "numero": 3,
            "nom": "Traitement",
            "statut": "✅ Réalisé (Session S01)",
            "date": datetime.now().strftime("%Y-%m-%d"),
            "responsable": "Data Steward IDA (MIDA507)",
            "actions": [
                "Suppression des durées aberrantes (>90 jours) : 0 cas",
                "Remplacement des filières manquantes par 'Non renseignée'",
                "Ajout du champ 'annee' calculé",
                "Vérification de l'encodage UTF-8"
            ],
            "outil": "Python pandas / Google Colab",
            "lacunes_identifiees": "Aucun clustering des valeurs proches (ex: cotes)"
        },
        {
            "numero": 4,
            "nom": "Analyse",
            "statut": "✅ Réalisé (Session S01)",
            "date": datetime.now().strftime("%Y-%m-%d"),
            "responsable": "Data Steward IDA (MIDA507)",
            "actions": [
                "Analyse des 7V du Big Data",
                "Identification des filières les plus actives",
                "Analyse de la saisonnalité des emprunts",
                "Tableau de bord de visualisation produit"
            ],
            "outil": "Python matplotlib / seaborn",
            "lacunes_identifiees": "Pas d'analyse prédictive (nécessiterait ML)"
        },
        {
            "numero": 5,
            "nom": "Publication",
            "statut": "🔄 En cours (Sessions S03–S06)",
            "date": "À venir",
            "responsable": "Data Steward IDA (MIDA507)",
            "actions": [
                "Rédaction de la fiche DCAT (session S04)",
                "Choix de la licence CC-BY (session S03)",
                "Publication sur portail CKAN Demo (session S06)",
                "Anonymisation des codes usagers (session S07)"
            ],
            "outil": "CKAN, JSON-LD Playground, DCAT Validator",
            "lacunes_identifiees": "Codes usagers non encore anonymisés"
        },
        {
            "numero": 6,
            "nom": "Préservation",
            "statut": "⏳ Planifié",
            "date": "À venir",
            "responsable": "DSI + Data Steward IDA",
            "actions": [
                "Migration vers CSV + backup DSpace institutionnel",
                "Génération des checksums SHA-256 de chaque fichier",
                "Plan de migration sur 10 ans (formats pérennes)"
            ],
            "outil": "DSpace, Python hashlib",
            "lacunes_identifiees": "Pas encore de backup institutionnel identifié"
        },
        {
            "numero": 7,
            "nom": "Réutilisation",
            "statut": "⏳ Planifié",
            "date": "À venir",
            "responsable": "Data Steward IDA",
            "actions": [
                "Mise à jour annuelle des métadonnées DCAT",
                "Collecte des retours d'usage des réutilisateurs",
                "Révision des licences si nécessaire"
            ],
            "outil": "CKAN API, email institutionnel",
            "lacunes_identifiees": "Pas encore de mécanisme de feedback"
        }
    ]
})

# Affichage formaté
print("╔══════════════════════════════════════════════════════════════╗")
print("║     JOURNAL DU CYCLE DE VIE — Bibliothèque UNG              ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"  Identifiant : {cycle_vie_journal['identifiant_jeu']}")
print(f"  Institution : {cycle_vie_journal['institution']}")
print(f"  Créé le     : {cycle_vie_journal['date_creation_journal']}")
print()
for etape in cycle_vie_journal["etapes"]:
    print(f"  Étape {etape['numero']} — {etape['nom'].upper()}")
    print(f"    Statut       : {etape['statut']}")
    print(f"    Responsable  : {etape['responsable']}")
    print(f"    Lacune IDA   : {etape['lacunes_identifiees']}")
    print()

# Sauvegarde
with open("MIDA507_S02_journal_cycle_vie.json", "w", encoding="utf-8") as f:
    json.dump(cycle_vie_journal, f, ensure_ascii=False, indent=2)
print("✅ Journal sauvegardé : MIDA507_S02_journal_cycle_vie.json")


In [ ]:
# ============================================================
# CELLULE 2.3 — Vérification de l'intégrité (Checksum SHA-256)
# ============================================================
# Un checksum est une "empreinte digitale" d'un fichier.
# Si le fichier est modifié, son checksum change.
# C'est l'outil de base de la préservation numérique.

def calculer_checksum(df):
    '''Calcule un checksum SHA-256 sur le contenu d'un DataFrame.'''
    contenu = df.to_csv(index=False).encode("utf-8")
    return hashlib.sha256(contenu).hexdigest()

checksum_original = calculer_checksum(df_base)

print("=" * 60)
print("ÉTAPE 6 — PRÉSERVATION : Calcul du checksum SHA-256")
print("=" * 60)
print(f"
Checksum du jeu de données original :")
print(f"  SHA-256 : {checksum_original}")
print(f"  (64 caractères hexadécimaux = empreinte unique du fichier)")

# Simuler une modification malveillante
df_modifie = df_base.copy()
df_modifie.loc[0, "duree_jours"] = 999  # Modification d'une valeur
checksum_modifie = calculer_checksum(df_modifie)

print(f"
Checksum après modification d'UNE SEULE valeur :")
print(f"  SHA-256 : {checksum_modifie}")
print(f"
  Identiques ? {'✅ OUI' if checksum_original == checksum_modifie else '❌ NON — modification détectée !'}")

print(f'''
💡 APPLICATION IDA — PRÉSERVATION :
   Avant d'archiver un fichier de données, le data steward calcule
   son checksum et le conserve séparément.
   À chaque vérification ultérieure, il recalcule le checksum
   et compare : si différent → le fichier a été altéré.
   
   C'est le principe de la chaîne de custody numérique —
   équivalent archivistique du paraphe sur chaque page d'un registre.
''')

# Sauvegarde du checksum de référence
registre_checksums = {
    "jeu_de_donnees": "MIDA507_S01_bibliotheque_ngaoundere_clean.csv",
    "checksum_sha256": checksum_original,
    "date_calcul": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "nb_lignes": len(df_base),
    "nb_colonnes": len(df_base.columns)
}
with open("MIDA507_S02_registre_checksums.json", "w") as f:
    json.dump(registre_checksums, f, ensure_ascii=False, indent=2)
print("✅ Registre de checksums sauvegardé.")


---
## PARTIE 3 · Évaluation des Principes FAIR

> **Rappel théorique :** Les principes FAIR (Findable, Accessible, Interoperable, Reusable) sont le standard international d'évaluation de la qualité des données. Dans cette partie, nous construisons un **évaluateur FAIR automatique** en Python et l'appliquons à notre jeu de données — puis à un jeu simulant des données gouvernementales africaines mal documentées.


In [ ]:
# ============================================================
# CELLULE 3.1 — Définir le profil FAIR d'un jeu de données
# ============================================================
# On crée une structure de profil FAIR pour notre jeu.
# En production, ce profil serait rempli depuis les métadonnées
# d'un portail CKAN ou d'un catalogue institutionnel.

def creer_profil_fair(
    # F — Findable
    identifiant_persistant=None,   # DOI, ARK, Handle...
    titre=None,
    mots_cles=None,
    catalogue_reference=None,      # URL du catalogue ou portail
    # A — Accessible
    url_telechargement=None,
    protocole_ouvert=None,         # "HTTPS", "HTTP", "FTP"...
    acces_sans_inscription=None,   # True / False
    politique_acces_documentee=None,
    # I — Interoperable
    format_fichier=None,           # "CSV", "JSON", "XLS", "PDF"...
    standard_metadonnees=None,     # "Dublin Core", "DCAT", "ISAD"...
    vocabulaire_controle=None,     # "AGROVOC", "RAMEAU", "Libre"...
    # R — Reusable
    licence_uri=None,              # URI de la licence CC
    provenance_documentee=None,    # True / False
    journal_transformations=None,  # True / False
    description_contexte=None,     # True / False
):
    return {
        "F": {
            "F1_identifiant_persistant": identifiant_persistant,
            "F2_titre_descriptif": titre,
            "F3_mots_cles": mots_cles,
            "F4_catalogue_reference": catalogue_reference,
        },
        "A": {
            "A1_url_telechargement": url_telechargement,
            "A1a_protocole_ouvert": protocole_ouvert,
            "A1b_acces_sans_inscription": acces_sans_inscription,
            "A2_politique_acces_documentee": politique_acces_documentee,
        },
        "I": {
            "I1_format_ouvert": format_fichier,
            "I2_standard_metadonnees": standard_metadonnees,
            "I3_vocabulaire_controle": vocabulaire_controle,
        },
        "R": {
            "R1_licence_uri": licence_uri,
            "R1a_provenance_documentee": provenance_documentee,
            "R1b_journal_transformations": journal_transformations,
            "R2_description_contexte": description_contexte,
        }
    }

print("✅ Fonction creer_profil_fair() définie.")
print("   Elle accepte 14 paramètres couvrant les 4 principes FAIR.")


In [ ]:
# ============================================================
# CELLULE 3.2 — Évaluateur FAIR automatique
# ============================================================
# Cette fonction calcule un score FAIR sur 100 points
# et génère des recommandations par principe.

def evaluer_fair(profil, nom_jeu="Jeu de données"):
    '''
    Évalue un profil FAIR et retourne un score et des recommandations.
    Chaque critère vaut entre 0 et 1. Score final sur 100.
    '''
    scores = {}
    recommandations = []

    # ── F — FINDABLE (25 points) ──
    f_scores = []

    # F1 : Identifiant persistant
    if profil["F"]["F1_identifiant_persistant"] and        any(p in str(profil["F"]["F1_identifiant_persistant"]).lower()
           for p in ["doi", "ark", "handle", "urn"]):
        f_scores.append(1.0)
    elif profil["F"]["F1_identifiant_persistant"]:
        f_scores.append(0.5)  # URL simple mais présente
        recommandations.append(
            "⚠️ [F1] L'identifiant n'est pas persistant (DOI/ARK recommandé). "
            "Alternative africaine gratuite : ARK (ark.bnf.fr).")
    else:
        f_scores.append(0.0)
        recommandations.append(
            "❌ [F1] Aucun identifiant persistant. "
            "Créer un ARK (gratuit) ou un DOI via DataCite.")

    # F2 : Titre descriptif
    if profil["F"]["F2_titre_descriptif"] and        len(str(profil["F"]["F2_titre_descriptif"])) >= 20:
        f_scores.append(1.0)
    elif profil["F"]["F2_titre_descriptif"]:
        f_scores.append(0.5)
        recommandations.append(
            "⚠️ [F2] Titre trop court. Inclure : sujet, zone géo, période. "
            "Ex: 'Emprunts documentaires — BU Ngaoundéré 2018-2023'")
    else:
        f_scores.append(0.0)
        recommandations.append("❌ [F2] Titre manquant. Champ obligatoire.")

    # F3 : Mots-clés
    if profil["F"]["F3_mots_cles"] and len(profil["F"]["F3_mots_cles"]) >= 5:
        f_scores.append(1.0)
    elif profil["F"]["F3_mots_cles"] and len(profil["F"]["F3_mots_cles"]) >= 2:
        f_scores.append(0.5)
        recommandations.append(
            "⚠️ [F3] Moins de 5 mots-clés. Ajouter des termes issus "
            "d'AGROVOC ou RAMEAU pour l'interopérabilité.")
    else:
        f_scores.append(0.0)
        recommandations.append(
            "❌ [F3] Aucun mot-clé. "
            "Ajouter au moins 5 mots-clés (dont 3 issus d'un vocabulaire contrôlé).")

    # F4 : Catalogue de référence
    if profil["F"]["F4_catalogue_reference"]:
        f_scores.append(1.0)
    else:
        f_scores.append(0.0)
        recommandations.append(
            "❌ [F4] Jeu non référencé dans un catalogue. "
            "Publier sur CKAN ou déclarer dans un catalogue institutionnel.")

    scores["F"] = round(np.mean(f_scores) * 25, 1)

    # ── A — ACCESSIBLE (25 points) ──
    a_scores = []

    if profil["A"]["A1_url_telechargement"]:
        a_scores.append(1.0)
    else:
        a_scores.append(0.0)
        recommandations.append("❌ [A1] Pas d'URL de téléchargement.")

    if profil["A"]["A1a_protocole_ouvert"] in ["HTTPS", "HTTP"]:
        a_scores.append(1.0)
    else:
        a_scores.append(0.0)
        recommandations.append(
            "❌ [A1a] Protocole absent ou non standard. Utiliser HTTPS.")

    if profil["A"]["A1b_acces_sans_inscription"] is True:
        a_scores.append(1.0)
    elif profil["A"]["A1b_acces_sans_inscription"] is False:
        a_scores.append(0.3)
        recommandations.append(
            "⚠️ [A1b] Inscription requise pour télécharger. "
            "Principe Sunlight n°4 non respecté. Accès libre recommandé.")
    else:
        a_scores.append(0.0)

    if profil["A"]["A2_politique_acces_documentee"]:
        a_scores.append(1.0)
    else:
        a_scores.append(0.0)
        recommandations.append(
            "❌ [A2] Politique d'accès non documentée. "
            "Préciser dans la fiche DCAT : ouvert / restreint / embargo.")

    scores["A"] = round(np.mean(a_scores) * 25, 1)

    # ── I — INTEROPERABLE (25 points) ──
    i_scores = []

    formats_ouverts = ["csv","json","xml","ods","rdf","turtle","geojson","parquet"]
    fmt = str(profil["I"]["I1_format_ouvert"] or "").lower()
    if fmt in formats_ouverts:
        i_scores.append(1.0)
    elif fmt in ["xls","xlsx"]:
        i_scores.append(0.4)
        recommandations.append(
            "⚠️ [I1] Format XLS/XLSX propriétaire. "
            "Exporter en CSV ou ODS pour l'interopérabilité.")
    else:
        i_scores.append(0.0)
        recommandations.append(
            "❌ [I1] Format non ouvert ou non renseigné. "
            "Utiliser CSV, JSON ou ODS.")

    standards_metadonnees = ["dublin core","dcat","isad","marc","schema.org",
                             "datacite","ddi","inspire"]
    std = str(profil["I"]["I2_standard_metadonnees"] or "").lower()
    if any(s in std for s in standards_metadonnees):
        i_scores.append(1.0)
    elif profil["I"]["I2_standard_metadonnees"]:
        i_scores.append(0.5)
        recommandations.append(
            f"⚠️ [I2] Standard '{profil['I']['I2_standard_metadonnees']}' "
            "non reconnu. Utiliser DCAT ou Dublin Core.")
    else:
        i_scores.append(0.0)
        recommandations.append(
            "❌ [I2] Aucun standard de métadonnées. "
            "Adopter DCAT (standard W3C pour les données) — session S04.")

    vocabulaires_controles = ["agrovoc","rameau","eurovoc","gemet","viaf",
                               "afristat","inspire","mesh","lcsh"]
    voc = str(profil["I"]["I3_vocabulaire_controle"] or "").lower()
    if any(v in voc for v in vocabulaires_controles):
        i_scores.append(1.0)
    elif profil["I"]["I3_vocabulaire_controle"] and "libre" not in voc:
        i_scores.append(0.5)
        recommandations.append(
            "⚠️ [I3] Vocabulaire non standard. "
            "Ajouter des URIs AGROVOC ou RAMEAU pour les mots-clés.")
    else:
        i_scores.append(0.0)
        recommandations.append(
            "❌ [I3] Pas de vocabulaire contrôlé. "
            "Utiliser AGROVOC (FAO) pour les données africaines.")

    scores["I"] = round(np.mean(i_scores) * 25, 1)

    # ── R — REUSABLE (25 points) ──
    r_scores = []

    licences_valides = ["creativecommons.org","opendatacommons.org",
                        "etalab.gouv.fr","data.gouv.fr/licence"]
    lic = str(profil["R"]["R1_licence_uri"] or "").lower()
    if any(l in lic for l in licences_valides):
        r_scores.append(1.0)
    elif profil["R"]["R1_licence_uri"] and len(lic) > 5:
        r_scores.append(0.5)
        recommandations.append(
            "⚠️ [R1] Licence mentionnée mais URI non standard. "
            "Utiliser l'URI complète : https://creativecommons.org/licenses/by/4.0/")
    else:
        r_scores.append(0.0)
        recommandations.append(
            "❌ [R1] Pas de licence. CRITIQUE : sans licence, "
            "la réutilisation est juridiquement impossible.")

    r_scores.append(1.0 if profil["R"]["R1a_provenance_documentee"] else 0.0)
    if not profil["R"]["R1a_provenance_documentee"]:
        recommandations.append(
            "❌ [R1a] Provenance non documentée. "
            "Préciser : qui a collecté ? Quand ? Comment ?")

    r_scores.append(1.0 if profil["R"]["R1b_journal_transformations"] else 0.0)
    if not profil["R"]["R1b_journal_transformations"]:
        recommandations.append(
            "⚠️ [R1b] Journal des transformations absent. "
            "Exporter l'historique OpenRefine ou le notebook Colab.")

    r_scores.append(1.0 if profil["R"]["R2_description_contexte"] else 0.0)
    if not profil["R"]["R2_description_contexte"]:
        recommandations.append(
            "❌ [R2] Contexte de collecte non décrit. "
            "Ajouter dans la fiche DCAT : méthode, population, couverture.")

    scores["R"] = round(np.mean(r_scores) * 25, 1)

    # Score total
    score_total = sum(scores.values())

    return {
        "nom": nom_jeu,
        "scores": scores,
        "score_total": round(score_total, 1),
        "niveau": ("🏆 Excellent (≥85)" if score_total >= 85
                   else "✅ Bon (≥65)" if score_total >= 65
                   else "⚠️ Passable (≥45)" if score_total >= 45
                   else "❌ Insuffisant (<45)"),
        "recommandations": recommandations,
        "nb_criteres_ok": sum(1 for v in scores.values() if v >= 18),
    }

print("✅ Fonction evaluer_fair() définie (14 critères, 100 points).")


In [ ]:
# ============================================================
# CELLULE 3.3 — Évaluation FAIR de notre jeu de bibliothèque
# ============================================================
# On évalue notre jeu de bibliothèque UNG DANS SON ÉTAT ACTUEL
# (avant la publication DCAT de la session S04)

profil_ung_actuel = creer_profil_fair(
    # F — Findable
    identifiant_persistant=None,          # ← Pas encore d'identifiant
    titre="Emprunts bibliothèque",        # ← Titre trop court
    mots_cles=["bibliothèque","emprunt"], # ← Seulement 2 mots-clés
    catalogue_reference=None,             # ← Pas encore publié

    # A — Accessible
    url_telechargement=None,              # ← Pas encore en ligne
    protocole_ouvert=None,
    acces_sans_inscription=None,
    politique_acces_documentee=None,

    # I — Interoperable
    format_fichier="CSV",                 # ← Bon format
    standard_metadonnees=None,            # ← Pas encore de DCAT
    vocabulaire_controle="Mots-clés libres",  # ← Pas de vocabulaire

    # R — Reusable
    licence_uri=None,                     # ← Pas encore de licence
    provenance_documentee=True,           # ← Journal cycle de vie créé
    journal_transformations=True,         # ← Notebook S01 = journal
    description_contexte=False            # ← Contexte non décrit
)

resultat_actuel = evaluer_fair(profil_ung_actuel, "BU Ngaoundéré — État actuel (avant S04)")

print("╔══════════════════════════════════════════════════════════════╗")
print("║   ÉVALUATION FAIR — BU Ngaoundéré AVANT publication DCAT    ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"  Score F (Findable)      : {resultat_actuel['scores']['F']:5.1f} / 25")
print(f"  Score A (Accessible)    : {resultat_actuel['scores']['A']:5.1f} / 25")
print(f"  Score I (Interoperable) : {resultat_actuel['scores']['I']:5.1f} / 25")
print(f"  Score R (Reusable)      : {resultat_actuel['scores']['R']:5.1f} / 25")
print(f"  ───────────────────────────────────────────")
print(f"  SCORE TOTAL             : {resultat_actuel['score_total']:5.1f} / 100")
print(f"  Niveau                  : {resultat_actuel['niveau']}")
print()
print("  RECOMMANDATIONS IDA :")
for rec in resultat_actuel["recommandations"]:
    print(f"    {rec}")


In [ ]:
# ============================================================
# CELLULE 3.4 — Comparaison FAIR : avant vs après les sessions
# ============================================================
# On simule le score FAIR qu'aura notre jeu APRÈS
# avoir appliqué les bonnes pratiques des sessions S03-S06.

profil_ung_cible = creer_profil_fair(
    # F — Findable (amélioré en S04)
    identifiant_persistant="ark:/67375/UNG-BIBLIO-2018-2023",
    titre="Emprunts documentaires — Bibliothèque Centrale Université de Ngaoundéré 2018-2023",
    mots_cles=["bibliothèque universitaire","emprunt","Cameroun",
               "documentation","éducation supérieure","Adamaoua",
               "data stewardship"],
    catalogue_reference="https://ckan.demo.org/dataset/ung-emprunts-2018-2023",

    # A — Accessible (amélioré en S06)
    url_telechargement="https://ckan.demo.org/dataset/ung-emprunts-2018-2023/ung_emprunts.csv",
    protocole_ouvert="HTTPS",
    acces_sans_inscription=True,
    politique_acces_documentee="Accès libre sans restriction — CC-BY 4.0",

    # I — Interoperable (amélioré en S04)
    format_fichier="CSV",
    standard_metadonnees="DCAT",
    vocabulaire_controle="AGROVOC (FAO) pour les thèmes disciplinaires",

    # R — Reusable (amélioré en S03 + S07)
    licence_uri="https://creativecommons.org/licenses/by/4.0/",
    provenance_documentee=True,
    journal_transformations=True,
    description_contexte=True
)

resultat_cible = evaluer_fair(profil_ung_cible, "BU Ngaoundéré — Cible après sessions S03-S06")

# ── Comparaison visuelle ──
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
principes = ["F
Findable", "A
Accessible", "I
Interoperable", "R
Reusable"]

scores_avant = list(resultat_actuel["scores"].values())
scores_apres = list(resultat_cible["scores"].values())
couleurs_avant = [PAL["gris"]] * 4
couleurs_apres = [PAL["ter2"], PAL["olive"], PAL["bleu"], PAL["ter3"]]

x = np.arange(4)
width = 0.35

# Graphique 1 : comparaison par principe
bars1 = axes[0].bar(x - width/2, scores_avant, width,
                    label=f"Avant (score: {resultat_actuel['score_total']}/100)",
                    color=PAL["gris"], alpha=0.8, edgecolor="white")
bars2 = axes[0].bar(x + width/2, scores_apres, width,
                    label=f"Après (score: {resultat_cible['score_total']}/100)",
                    color=PAL["ter2"], alpha=0.9, edgecolor="white")
axes[0].set_title("Scores FAIR par principe
Avant vs Après les sessions S03-S06",
                  fontweight="bold", fontsize=12)
axes[0].set_xticks(x)
axes[0].set_xticklabels(principes)
axes[0].set_ylabel("Score /25")
axes[0].set_ylim(0, 27)
axes[0].axhline(y=18, color="red", linestyle="--", alpha=0.4, label="Seuil acceptable (72%)")
axes[0].legend(fontsize=9)
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f"{bar.get_height():.1f}", ha="center", fontsize=9)
for bar in bars2:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f"{bar.get_height():.1f}", ha="center", fontsize=9)

# Graphique 2 : radar / jauge du score total
categories = ["Avant S04
(état actuel)", "Cible
(après S03-S06)"]
valeurs = [resultat_actuel["score_total"], resultat_cible["score_total"]]
couleurs_barre = [PAL["gris"], PAL["olive"]]
barres = axes[1].barh(categories, valeurs, color=couleurs_barre,
                       edgecolor="white", height=0.4)
axes[1].set_xlim(0, 100)
axes[1].set_title("Score FAIR Total /100
Bibliothèque UNG",
                  fontweight="bold", fontsize=12)
axes[1].axvline(x=45, color="red",    linestyle="--", alpha=0.5, label="Insuffisant (<45)")
axes[1].axvline(x=65, color="orange", linestyle="--", alpha=0.5, label="Bon (≥65)")
axes[1].axvline(x=85, color="green",  linestyle="--", alpha=0.5, label="Excellent (≥85)")
for bar, val in zip(barres, valeurs):
    axes[1].text(val + 1, bar.get_y() + bar.get_height()/2,
                 f"{val}/100 — {('Insuffisant' if val < 45 else 'Passable' if val < 65 else 'Bon' if val < 85 else 'Excellent')}",
                 va="center", fontsize=11, fontweight="bold")
axes[1].legend(fontsize=8)

plt.suptitle("Évaluation FAIR — Impact des sessions MIDA507 sur la qualité du jeu de données",
             fontsize=13, fontweight="bold", color=PAL["ter1"], y=1.02)
plt.tight_layout()
plt.show()

print(f"
📈 Progression FAIR prévue : {resultat_actuel['score_total']} → {resultat_cible['score_total']} points")
print(f"   (+{resultat_cible['score_total'] - resultat_actuel['score_total']} points en appliquant les bonnes pratiques des sessions S03-S06)")


In [ ]:
# ============================================================
# CELLULE 3.5 — Évaluation FAIR d'un portail africain réel (simulé)
# ============================================================
# On simule l'évaluation d'un jeu de données de type INS Cameroun
# dans son état typique (XLS, sans licence, métadonnées minimales)

profil_ins_typique = creer_profil_fair(
    # Situation typique d'un portail statistique africain
    identifiant_persistant="https://statistics.cm/data/emploi2023.xls",  # URL simple
    titre="Emploi 2023",                         # Titre trop court
    mots_cles=["emploi","chômage"],              # Seulement 2
    catalogue_reference=None,                    # Pas de catalogue

    url_telechargement="https://statistics.cm/data/emploi2023.xls",
    protocole_ouvert="HTTPS",
    acces_sans_inscription=False,               # Inscription requise
    politique_acces_documentee=None,

    format_fichier="XLS",                       # Format propriétaire
    standard_metadonnees=None,
    vocabulaire_controle=None,

    licence_uri=None,                           # PAS DE LICENCE
    provenance_documentee=False,
    journal_transformations=False,
    description_contexte=False
)

profil_data_gouv_bj = creer_profil_fair(
    # data.gouv.bj : meilleur portail francophone africain
    identifiant_persistant="https://data.gouv.bj/dataset/effectifs-scolaires-2023",
    titre="Effectifs scolaires par commune et niveau d'enseignement — Bénin 2022-2023",
    mots_cles=["éducation","école","effectifs","Bénin","2023",
               "commune","enseignement primaire"],
    catalogue_reference="https://data.gouv.bj/dataset/effectifs-scolaires-2023",

    url_telechargement="https://data.gouv.bj/dataset/effectifs/download/effectifs_2023.csv",
    protocole_ouvert="HTTPS",
    acces_sans_inscription=True,
    politique_acces_documentee="Licence Ouverte Etalab v2 — accès libre",

    format_fichier="CSV",
    standard_metadonnees="Dublin Core",
    vocabulaire_controle="Mots-clés libres thématiques",

    licence_uri="https://www.etalab.gouv.fr/wp-content/uploads/2017/04/ETALAB-Licence-Ouverte-v2.0.pdf",
    provenance_documentee=True,
    journal_transformations=False,
    description_contexte=True
)

r_ins  = evaluer_fair(profil_ins_typique,   "INS Cameroun — Situation typique")
r_bj   = evaluer_fair(profil_data_gouv_bj,  "data.gouv.bj — Référence francophone")
r_cible = resultat_cible

# ── Tableau comparatif ──
print("═" * 65)
print("COMPARAISON FAIR — 3 portails africains")
print("═" * 65)
print(f"{'Portail':<35} {'F':>5} {'A':>5} {'I':>5} {'R':>5} {'TOTAL':>7}")
print("─" * 65)
for r in [r_ins, r_bj, r_cible]:
    s = r["scores"]
    print(f"{r['nom']:<35} {s['F']:>5.1f} {s['A']:>5.1f} {s['I']:>5.1f} "
          f"{s['R']:>5.1f} {r['score_total']:>7.1f}")
print("─" * 65)
print(f"{'Maximum possible':<35} {'25':>5} {'25':>5} {'25':>5} {'25':>5} {'100':>7}")

print(f'''
💡 ANALYSE IDA :
   L'INS Cameroun typique obtient un score faible principalement à cause
   de 3 problèmes corrigeables : format XLS → CSV (gratuit), 
   licence absente → CC-BY (5 minutes), inscription obligatoire → supprimer.
   
   data.gouv.bj est le modèle de référence pour l'Afrique francophone :
   format CSV, accès libre, licence Etalab, provenance documentée.
   
   Votre jeu après MIDA507 atteindra un score similaire.
''')


---
## PARTIE 4 · Construire un Data Management Plan (DMP)

> **Rappel théorique :** Un DMP décrit comment les données d'un projet seront collectées, documentées, stockées, partagées et préservées. Il est exigé par les bailleurs de fonds (AFD, IDRC, Horizon Europe). Dans cette partie, nous construisons un DMP structuré en Python et l'exportons en JSON et en Markdown.


In [ ]:
# ============================================================
# CELLULE 4.1 — Classe DMP : structure en 7 sections
# ============================================================
# On crée une classe Python représentant un DMP.
# En production, on utiliserait DMPOnline (dmponline.dcc.ac.uk)
# qui génère ce type de document automatiquement.

class DataManagementPlan:
    '''Représente un Data Management Plan structuré selon les 7 sections DCC.'''

    def __init__(self, titre_projet, institution, bailleur, date_debut, duree_mois):
        self.meta = {
            "titre_projet":  titre_projet,
            "institution":   institution,
            "bailleur":      bailleur,
            "date_debut":    date_debut,
            "duree_mois":    duree_mois,
            "data_steward":  None,
            "date_dmp":      datetime.now().strftime("%Y-%m-%d"),
            "version":       "1.0"
        }
        self.sections = {s: {} for s in [
            "1_description_donnees",
            "2_documentation_metadonnees",
            "3_stockage_securite",
            "4_aspects_legaux_ethiques",
            "5_partage_publication",
            "6_preservation_long_terme",
            "7_responsabilites"
        ]}

    def set_responsable(self, nom, email, role="Data Steward IDA"):
        self.meta["data_steward"] = {"nom": nom, "email": email, "role": role}

    def section_1_description(self, types_donnees, formats, volume_estime,
                               sources, frequence_creation):
        self.sections["1_description_donnees"] = {
            "types_donnees_produits":  types_donnees,
            "formats_utilises":        formats,
            "volume_estime":           volume_estime,
            "sources_des_donnees":     sources,
            "frequence_de_creation":   frequence_creation,
            "competence_IDA_mobilisee": "Classification, inventaire, archivage"
        }

    def section_2_documentation(self, standard_metadonnees, champs_obligatoires,
                                  niveau_granularite, outil_catalogue):
        self.sections["2_documentation_metadonnees"] = {
            "standard_metadonnees":     standard_metadonnees,
            "champs_obligatoires":      champs_obligatoires,
            "niveau_granularite":       niveau_granularite,
            "outil_catalogue":          outil_catalogue,
            "competence_IDA_mobilisee": "Catalogage, indexation, normes DCAT/Dublin Core"
        }

    def section_3_stockage(self, infrastructure, politique_backup,
                            controle_acces, hebergeur):
        self.sections["3_stockage_securite"] = {
            "infrastructure":          infrastructure,
            "politique_backup":        politique_backup,
            "controle_acces":          controle_acces,
            "hebergeur_recommande":    hebergeur,
            "competence_IDA_mobilisee": "Conservation, politique de rétention, sécurité"
        }

    def section_4_legal_ethique(self, base_legale_collecte, licence_publication,
                                  donnees_personnelles, procedure_anonymisation,
                                  rgpd_applicable):
        self.sections["4_aspects_legaux_ethiques"] = {
            "base_legale_collecte":     base_legale_collecte,
            "licence_publication":      licence_publication,
            "donnees_personnelles":     donnees_personnelles,
            "procedure_anonymisation":  procedure_anonymisation,
            "rgpd_ou_loi_africaine":    rgpd_applicable,
            "competence_IDA_mobilisee": "Droit de l'information, RGPD, éthique professionnelle"
        }

    def section_5_partage(self, portail_publication, date_ouverture,
                           format_diffusion, embargo):
        self.sections["5_partage_publication"] = {
            "portail_publication":     portail_publication,
            "date_ouverture_prevue":   date_ouverture,
            "format_de_diffusion":     format_diffusion,
            "embargo_eventuel":        embargo,
            "competence_IDA_mobilisee": "Diffusion documentaire, portails open data"
        }

    def section_6_preservation(self, formats_perennes, entrepot_certifie,
                                 plan_migration, duree_conservation):
        self.sections["6_preservation_long_terme"] = {
            "formats_perennes":        formats_perennes,
            "entrepot_certifie":       entrepot_certifie,
            "plan_migration":          plan_migration,
            "duree_conservation":      duree_conservation,
            "competence_IDA_mobilisee": "Archivage pérenne, préservation numérique"
        }

    def section_7_responsabilites(self, equipe, calendrier_revision,
                                   budget_gestion_donnees):
        self.sections["7_responsabilites"] = {
            "equipe_et_roles":         equipe,
            "calendrier_revision_dmp": calendrier_revision,
            "budget_gestion_donnees":  budget_gestion_donnees,
            "competence_IDA_mobilisee": "Gouvernance, coordination, formation"
        }

    def to_dict(self):
        return {"meta": self.meta, "sections": self.sections}

    def to_markdown(self):
        '''Génère une version Markdown lisible du DMP.'''
        md = f"# Data Management Plan
"
        md += f"## {self.meta['titre_projet']}

"
        md += f"**Institution :** {self.meta['institution']}  
"
        md += f"**Bailleur :** {self.meta['bailleur']}  
"
        md += f"**Début :** {self.meta['date_debut']} | Durée : {self.meta['duree_mois']} mois  
"
        if self.meta["data_steward"]:
            ds = self.meta["data_steward"]
            md += f"**Data Steward IDA :** {ds['nom']} ({ds['email']})  
"
        md += f"**Version DMP :** {self.meta['version']} — {self.meta['date_dmp']}

"
        md += "---

"
        for section_key, section_val in self.sections.items():
            titre = section_key.replace("_", " ").replace("1 ", "1. ").replace(
                "2 ", "2. ").replace("3 ", "3. ").replace("4 ", "4. ").replace(
                "5 ", "5. ").replace("6 ", "6. ").replace("7 ", "7. ").upper()
            md += f"## {titre}

"
            if section_val:
                for k, v in section_val.items():
                    if k == "competence_IDA_mobilisee":
                        md += f"> 💼 **Compétence IDA :** {v}

"
                    elif isinstance(v, list):
                        md += f"**{k.replace('_',' ').capitalize()} :**
"
                        for item in v:
                            md += f"- {item}
"
                        md += "
"
                    else:
                        md += f"**{k.replace('_',' ').capitalize()} :** {v}  
"
                md += "
"
            else:
                md += "*Section à compléter.*

"
        return md

print("✅ Classe DataManagementPlan définie (7 sections, export JSON + Markdown).")


In [ ]:
# ============================================================
# CELLULE 4.2 — Rédiger le DMP de notre projet bibliothèque
# ============================================================

dmp = DataManagementPlan(
    titre_projet = "Valorisation open data des emprunts documentaires — BU Ngaoundéré",
    institution  = "Université de Ngaoundéré — Bibliothèque Centrale",
    bailleur     = "Carnegie Corporation of New York",
    date_debut   = "2024-09-01",
    duree_mois   = 18
)

dmp.set_responsable(
    nom   = "Data Steward IDA — MIDA507",
    email = "data-steward@univ-ngaoundere.cm",
    role  = "Data Steward et Responsable Publication Open Data"
)

dmp.section_1_description(
    types_donnees      = ["Enregistrements d'emprunt (CSV tabulaire)",
                          "Métadonnées des documents (CSV)",
                          "Journaux de nettoyage (JSON)",
                          "Notebooks d'analyse (IPYNB)"],
    formats            = ["CSV UTF-8 (format principal)",
                          "JSON (métadonnées DCAT)",
                          "PDF/A (rapports)"],
    volume_estime      = "5 000 enregistrements/an (~2 Mo/an CSV)",
    sources            = "Export mensuel du SIGB PMB de la Bibliothèque",
    frequence_creation = "Annuelle (export consolidé chaque 31 décembre)"
)

dmp.section_2_documentation(
    standard_metadonnees  = "DCAT W3C + Dublin Core",
    champs_obligatoires   = ["dct:title","dct:description","dcat:keyword",
                              "dct:publisher","dct:issued","dct:license",
                              "dct:spatial","dct:temporal","dcat:distribution"],
    niveau_granularite    = "Un jeu de données par année civile + 1 jeu consolidé",
    outil_catalogue       = "CKAN Demo puis portail institutionnel CKAN"
)

dmp.section_3_stockage(
    infrastructure  = "Serveur local PMB + backup Google Drive institutionnel",
    politique_backup = "Backup quotidien automatique + backup mensuel hors site",
    controle_acces  = "Accès lecture : public (CSV publié) | Accès écriture : data steward uniquement",
    hebergeur       = "CKAN hébergé chez Africa Data Centres (Johannesburg) — souveraineté africaine"
)

dmp.section_4_legal_ethique(
    base_legale_collecte    = "Règlement intérieur de la bibliothèque + accord institutionnel",
    licence_publication     = "Creative Commons CC-BY 4.0",
    donnees_personnelles    = "Codes usagers présents — anonymisation OBLIGATOIRE avant publication",
    procedure_anonymisation = "Suppression des codes usagers + généralisation filière → domaine",
    rgpd_applicable         = "Loi camerounaise n°2010/012 sur la cybersécurité et la cybercriminalité"
)

dmp.section_5_partage(
    portail_publication = "CKAN institutionnel Université de Ngaoundéré",
    date_ouverture_prevue = "2025-03-01 (18 mois après début du projet)",
    format_diffusion    = "CSV UTF-8 + ODS + JSON-LD DCAT",
    embargo             = "Aucun embargo — publication immédiate après anonymisation"
)

dmp.section_6_preservation(
    formats_perennes    = ["CSV UTF-8 (10 ans+)", "JSON (10 ans+)", "PDF/A (permanent)"],
    entrepot_certifie   = "Zenodo (CERN) pour l'archivage de long terme des versions annuelles",
    plan_migration      = "Révision des formats tous les 5 ans — migration si format obsolète",
    duree_conservation  = "10 ans après la dernière publication"
)

dmp.section_7_responsabilites(
    equipe = [
        {"role": "Data Steward IDA", "taches": "Nettoyage, DCAT, publication, maintenance"},
        {"role": "Direction Bibliothèque", "taches": "Validation politique, décisions d'accès"},
        {"role": "DSI Université", "taches": "Infrastructure, backup, hébergement"}
    ],
    calendrier_revision  = "Révision du DMP tous les 6 mois ou lors de changements majeurs",
    budget_gestion_donnees = "1 500 €/an (hébergement CKAN + formation data steward)"
)

# Affichage résumé
d = dmp.to_dict()
print("╔══════════════════════════════════════════════════════════════╗")
print("║             DATA MANAGEMENT PLAN — Résumé                   ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"  Projet      : {d['meta']['titre_projet']}")
print(f"  Institution : {d['meta']['institution']}")
print(f"  Bailleur    : {d['meta']['bailleur']}")
print(f"  Durée       : {d['meta']['duree_mois']} mois")
print(f"  Version     : {d['meta']['version']} ({d['meta']['date_dmp']})")
print()
for sec_key, sec_val in d["sections"].items():
    statut = "✅ Complète" if sec_val else "⏳ À compléter"
    print(f"  Section {sec_key[:1]}  : {sec_key[2:].replace('_',' ').capitalize()[:40]} → {statut}")
print("╚══════════════════════════════════════════════════════════════╝")


In [ ]:
# ============================================================
# CELLULE 4.3 — Export du DMP en JSON et Markdown
# ============================================================

# Export JSON
dmp_dict = dmp.to_dict()
with open("MIDA507_S02_DMP_BU_Ngaoundere.json", "w", encoding="utf-8") as f:
    json.dump(dmp_dict, f, ensure_ascii=False, indent=2)
print("✅ DMP exporté en JSON : MIDA507_S02_DMP_BU_Ngaoundere.json")

# Export Markdown
dmp_md = dmp.to_markdown()
with open("MIDA507_S02_DMP_BU_Ngaoundere.md", "w", encoding="utf-8") as f:
    f.write(dmp_md)
print("✅ DMP exporté en Markdown : MIDA507_S02_DMP_BU_Ngaoundere.md")

# Aperçu du Markdown
print("
─── APERÇU DU DMP EN MARKDOWN ───────────────────────────")
print(dmp_md[:1200])
print("... [suite dans le fichier complet]")


---
## PARTIE 5 · Documenter le Data Lineage (Traçabilité des Transformations)

> **Rappel théorique :** Le data lineage est la documentation complète de l'origine et des transformations subies par chaque donnée. C'est l'équivalent data de la chaîne de custody en archivistique. Sans data lineage, il est impossible d'évaluer la fiabilité d'une analyse.


In [ ]:
# ============================================================
# CELLULE 5.1 — Système de journalisation des transformations
# ============================================================
# On crée un journal automatique des transformations.
# Chaque modification du jeu de données est enregistrée
# avec : qui, quand, quoi, pourquoi.

class JournalTransformations:
    '''Journal de traçabilité des transformations d'un jeu de données.'''

    def __init__(self, nom_jeu, description_initiale):
        self.journal = {
            "jeu_de_donnees":         nom_jeu,
            "description_initiale":   description_initiale,
            "date_initialisation":    datetime.now().strftime("%Y-%m-%d %H:%M"),
            "transformations":        [],
            "etat_courant": {
                "nb_lignes": None,
                "nb_colonnes": None,
                "checksum": None
            }
        }

    def enregistrer(self, df_avant, df_apres, action, raison,
                    outil, responsable="Data Steward IDA"):
        '''Enregistre une transformation avec statistiques avant/après.'''
        entree = {
            "id": len(self.journal["transformations"]) + 1,
            "date":        datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "responsable": responsable,
            "outil":       outil,
            "action":      action,
            "raison_ида":  raison,
            "avant": {
                "nb_lignes":   len(df_avant),
                "nb_colonnes": len(df_avant.columns),
                "manquants":   int(df_avant.isnull().sum().sum()),
                "checksum":    hashlib.sha256(
                    df_avant.to_csv(index=False).encode()).hexdigest()[:16] + "..."
            },
            "apres": {
                "nb_lignes":   len(df_apres),
                "nb_colonnes": len(df_apres.columns),
                "manquants":   int(df_apres.isnull().sum().sum()),
                "checksum":    hashlib.sha256(
                    df_apres.to_csv(index=False).encode()).hexdigest()[:16] + "..."
            },
            "delta": {
                "lignes_modifiees": len(df_avant) - len(df_apres),
                "colonnes_ajoutees": len(df_apres.columns) - len(df_avant.columns),
                "manquants_corriges": int(df_avant.isnull().sum().sum()) - int(df_apres.isnull().sum().sum())
            }
        }
        self.journal["transformations"].append(entree)
        # Mise à jour de l'état courant
        self.journal["etat_courant"] = {
            "nb_lignes": len(df_apres),
            "nb_colonnes": len(df_apres.columns),
            "checksum": hashlib.sha256(
                df_apres.to_csv(index=False).encode()).hexdigest()
        }

    def afficher_resume(self):
        print(f"JOURNAL DE TRAÇABILITÉ — {self.journal['jeu_de_donnees']}")
        print(f"Initialisé le : {self.journal['date_initialisation']}")
        print(f"Nb transformations : {len(self.journal['transformations'])}")
        print()
        for t in self.journal["transformations"]:
            print(f"  Transformation #{t['id']} — {t['date']}")
            print(f"    Action      : {t['action']}")
            print(f"    Raison IDA  : {t['raison_ида']}")
            print(f"    Outil       : {t['outil']}")
            print(f"    Avant → Après : {t['avant']['nb_lignes']} → {t['apres']['nb_lignes']} lignes")
            print(f"    Manquants corrigés : {t['delta']['manquants_corriges']}")
            print()

    def to_json(self, fichier):
        with open(fichier, "w", encoding="utf-8") as f:
            json.dump(self.journal, f, ensure_ascii=False, indent=2)
        print(f"✅ Journal sauvegardé : {fichier}")

print("✅ Classe JournalTransformations définie.")


In [ ]:
# ============================================================
# CELLULE 5.2 — Application du data lineage sur notre jeu
# ============================================================
# On applique une série de transformations sur le jeu de données
# et on enregistre chaque étape dans le journal.

journal = JournalTransformations(
    nom_jeu              = "MIDA507-BIBLIO-UNG-2018-2023",
    description_initiale = "Export brut du SIGB PMB — Bibliothèque UNG"
)

df_courant = df_base.copy()

# ── TRANSFORMATION 1 : Standardisation des noms de colonnes ──
df_t1 = df_courant.copy()
df_t1.columns = [c.lower().replace(" ", "_") for c in df_t1.columns]
journal.enregistrer(
    df_avant     = df_courant,
    df_apres     = df_t1,
    action       = "Normalisation des noms de colonnes (minuscules + underscores)",
    raison       = "Conformité DCAT : les noms de colonnes ne doivent pas contenir "
                   "d'espaces ni de majuscules pour l'interopérabilité machine",
    outil        = "Python pandas — df.columns.str.lower()"
)
df_courant = df_t1
print(f"T1 appliquée — Colonnes : {list(df_courant.columns)}")

# ── TRANSFORMATION 2 : Ajout d'un identifiant ARK simulé ──
df_t2 = df_courant.copy()
df_t2.insert(0, "ark_id",
    [f"ark:/67375/UNG-{str(i+1).zfill(6)}" for i in range(len(df_t2))])
journal.enregistrer(
    df_avant     = df_courant,
    df_apres     = df_t2,
    action       = "Ajout d'un identifiant persistant ARK à chaque enregistrement",
    raison       = "Principe FAIR F1 : chaque enregistrement doit avoir "
                   "un identifiant unique et persistant pour être citable",
    outil        = "Python — pattern 'ark:/[NAAN]/[institution]-[id]'"
)
df_courant = df_t2
print(f"T2 appliquée — Colonne ark_id ajoutée : {df_courant['ark_id'].iloc[0]}")

# ── TRANSFORMATION 3 : Ajout des métadonnées de langue ISO 639 ──
mapping_langue_iso = {
    "Français":                "fra",
    "Anglais":                 "eng",
    "Bilingue FR/EN":          "fra+eng",
    "Arabe":                   "ara",
    "Autres langues africaines":"mul"  # 'mul' = multiple languages (ISO 639)
}
df_t3 = df_courant.copy()
df_t3["langue_iso639"] = df_t3["langue_document"].map(mapping_langue_iso)
journal.enregistrer(
    df_avant     = df_courant,
    df_apres     = df_t3,
    action       = "Ajout du code langue ISO 639-3",
    raison       = "Principe FAIR I1 : utiliser des codes standardisés ISO plutôt "
                   "que des libellés libres pour l'interopérabilité internationale",
    outil        = "Python pandas — .map() avec dictionnaire de correspondance"
)
df_courant = df_t3
print(f"T3 appliquée — Codes ISO: {df_courant['langue_iso639'].value_counts().to_dict()}")

# ── TRANSFORMATION 4 : Pseudonymisation des codes usagers ──
df_t4 = df_courant.copy()
# On remplace chaque code usager par un hash court (irréversible)
# Note : en S07, on verra la k-anonymité plus complète
df_t4["code_usager_anon"] = df_t4["code_usager"].apply(
    lambda x: "USR-" + hashlib.md5(str(x).encode()).hexdigest()[:8].upper()
)
df_t4 = df_t4.drop(columns=["code_usager"])
journal.enregistrer(
    df_avant     = df_courant,
    df_apres     = df_t4,
    action       = "Pseudonymisation des codes usagers par hachage MD5 tronqué",
    raison       = "RGPD / Loi 2010/012 Cameroun : les codes usagers sont des données "
                   "personnelles indirectement identifiantes — pseudonymisation "
                   "avant publication (anonymisation complète en S07)",
    outil        = "Python hashlib.md5() — irréversible sans table de correspondance"
)
df_courant = df_t4
print(f"T4 appliquée — Exemple avant: USR0042 | après: {df_courant['code_usager_anon'].iloc[0]}")

# ── Affichage du journal ──
print()
journal.afficher_resume()

# Sauvegarde
journal.to_json("MIDA507_S02_journal_transformations.json")
df_courant.to_csv("MIDA507_S02_bibliotheque_ung_enrichie.csv",
                  index=False, encoding="utf-8-sig")
print(f"
✅ Jeu enrichi sauvegardé : {len(df_courant)} lignes | {len(df_courant.columns)} colonnes")


In [ ]:
# ============================================================
# CELLULE 5.3 — Visualisation du data lineage
# ============================================================
# On affiche un schéma visuel du flux de transformations
# appliquées à notre jeu de données.

fig, ax = plt.subplots(figsize=(16, 5))
ax.set_xlim(-0.5, 16); ax.set_ylim(-1.5, 3.5)
ax.axis("off")

etapes_lineage = [
    ("SOURCE
PMB (SIGB)", 0.8,  PAL["gris"],  "Export CSV brut
5 000 lignes
10 colonnes"),
    ("T1
Norm. colonnes",  3.2,  PAL["ter3"],  "Colonnes standardisées
minuscules+underscore
IDA : conformité DCAT"),
    ("T2
ARK IDs",         5.6,  PAL["ter2"],  "Identifiants persistants
arc:/67375/UNG-...
IDA : principe FAIR F1"),
    ("T3
ISO 639",         8.0,  PAL["olive"], "Codes langue ISO
fra, eng, ara, mul
IDA : interopérabilité"),
    ("T4
Pseudonymisation",10.4, PAL["bleu"],  "Codes usagers hachés
protection données
IDA : RGPD/Loi cam."),
    ("RÉSULTAT
Jeu enrichi",12.8, PAL["ter1"], "5 000 lignes
12 colonnes
Prêt pour DCAT S04"),
]

for nom, x, couleur, details in etapes_lineage:
    # Boîte principale
    rect = plt.FancyBboxPatch((x, 1.5), 2.2, 1.2,
                               boxstyle="round,pad=0.05",
                               facecolor=couleur, edgecolor="white",
                               linewidth=2, zorder=3)
    ax.add_patch(rect)
    ax.text(x + 1.1, 2.1, nom, ha="center", va="center",
            fontsize=9, fontweight="bold", color="white", zorder=4)

    # Détails en dessous
    rect_det = plt.FancyBboxPatch((x, -0.2), 2.2, 1.4,
                                   boxstyle="round,pad=0.05",
                                   facecolor="white", edgecolor=couleur,
                                   linewidth=1.5, alpha=0.9, zorder=3)
    ax.add_patch(rect_det)
    ax.text(x + 1.1, 0.5, details, ha="center", va="center",
            fontsize=7.5, color=PAL["ter1"], zorder=4)

    # Flèche vers la suivante
    if x < 12.8:
        ax.annotate("", xy=(x + 2.4, 2.1), xytext=(x + 2.2, 2.1),
                    arrowprops=dict(arrowstyle="->",
                                   color=PAL["gris"], lw=2.5), zorder=5)

    # Connecteur vertical
    ax.annotate("", xy=(x + 1.1, 1.5), xytext=(x + 1.1, 1.2),
                arrowprops=dict(arrowstyle="-",
                               color=couleur, lw=1.5), zorder=5)

ax.set_title(
    "DATA LINEAGE — Bibliothèque UNG
Traçabilité des 4 transformations appliquées",
    fontsize=14, fontweight="bold", color=PAL["ter1"], pad=10
)
ax.text(8, -1.2,
        "Chaque transformation est enregistrée dans le journal JSON → "
        "traçabilité complète, reproductibilité garantie",
        ha="center", fontsize=9, color=PAL["gris"], style="italic")

plt.tight_layout()
plt.show()
print("
✅ Schéma de data lineage affiché.")


---
## PARTIE 6 · Exercices et Questions de Réflexion

> Complétez les zones **`[À COMPLÉTER]`**. Ces exercices constituent votre **livrable de participation** pour la session 02.


In [ ]:
# ============================================================
# EXERCICE 1 — Évaluer FAIR votre propre jeu de données
# ============================================================
# CONSIGNE : Créez le profil FAIR d'UN jeu de données
# de votre institution réelle ou fictive.
# Calculez son score et listez les 3 recommandations prioritaires.

mon_profil_fair = creer_profil_fair(
    # F — FINDABLE
    identifiant_persistant = "[À COMPLÉTER : URL ou ARK ou None]",
    titre                  = "[À COMPLÉTER : titre descriptif]",
    mots_cles              = ["[mot 1]","[mot 2]"],  # Au moins 2
    catalogue_reference    = "[À COMPLÉTER : URL ou None]",

    # A — ACCESSIBLE
    url_telechargement      = "[À COMPLÉTER : URL ou None]",
    protocole_ouvert        = "[À COMPLÉTER : 'HTTPS' ou None]",
    acces_sans_inscription  = None,  # True ou False
    politique_acces_documentee = "[À COMPLÉTER ou None]",

    # I — INTEROPERABLE
    format_fichier         = "[À COMPLÉTER : 'CSV', 'XLS', 'PDF', 'JSON'...]",
    standard_metadonnees   = "[À COMPLÉTER : 'Dublin Core', 'DCAT', None...]",
    vocabulaire_controle   = "[À COMPLÉTER : 'AGROVOC', 'RAMEAU', 'Mots-clés libres'...]",

    # R — REUSABLE
    licence_uri            = "[À COMPLÉTER : URI CC ou None]",
    provenance_documentee  = None,  # True ou False
    journal_transformations = None,  # True ou False
    description_contexte   = None   # True ou False
)

mon_resultat = evaluer_fair(mon_profil_fair, "Mon institution — [NOM]")

print(f"SCORE FAIR — {mon_resultat['nom']}")
print(f"  F : {mon_resultat['scores']['F']}/25")
print(f"  A : {mon_resultat['scores']['A']}/25")
print(f"  I : {mon_resultat['scores']['I']}/25")
print(f"  R : {mon_resultat['scores']['R']}/25")
print(f"  TOTAL : {mon_resultat['score_total']}/100 — {mon_resultat['niveau']}")
print()
print("3 RECOMMANDATIONS PRIORITAIRES :")
for rec in mon_resultat["recommandations"][:3]:
    print(f"  {rec}")


In [ ]:
# ============================================================
# EXERCICE 2 — Compléter une section du DMP
# ============================================================
# CONSIGNE : Complétez la section 4 (Aspects légaux et éthiques)
# du DMP pour UN de ces deux cas :
# CAS A : Jeu de données de recensement scolaire de votre pays
# CAS B : Jeu de données d'enquête santé avec noms de patients

# Choisissez votre cas : "A" ou "B"
cas_choisi = "[À COMPLÉTER : 'A' ou 'B']"

if cas_choisi == "A":
    print("CAS A — Recensement scolaire")
    base_legale   = "[À COMPLÉTER : quelle loi autorise cette collecte ?]"
    licence       = "[À COMPLÉTER : quelle licence recommandez-vous ? Pourquoi ?]"
    donnees_perso = "[À COMPLÉTER : y a-t-il des données personnelles ? Lesquelles ?]"
    procedure_ano = "[À COMPLÉTER : quelle procédure d'anonymisation si nécessaire ?]"

elif cas_choisi == "B":
    print("CAS B — Enquête santé avec noms de patients")
    base_legale   = "[À COMPLÉTER : consentement éclairé ? Loi nationale ?]"
    licence       = "[À COMPLÉTER : peut-on publier ces données ? Sous quelle licence ?]"
    donnees_perso = "[À COMPLÉTER : quelles données sont personnelles et sensibles ?]"
    procedure_ano = "[À COMPLÉTER : suppression ? agrégation ? k-anonymité ? Niveau k ?]"

else:
    print("⚠️ Veuillez choisir 'A' ou 'B' dans la variable cas_choisi.")
    base_legale = licence = donnees_perso = procedure_ano = "Non défini"

print(f"  Base légale    : {base_legale}")
print(f"  Licence        : {licence}")
print(f"  Données perso  : {donnees_perso}")
print(f"  Anonymisation  : {procedure_ano}")

print('''
❓ QUESTION DE RÉFLEXION :
   Dans le CAS B (données de santé), la loi camerounaise 2010/012 
   vous impose-t-elle d'anonymiser AVANT la publication ?
   Existe-t-il une autorité de contrôle des données personnelles
   au Cameroun à qui vous devriez déclarer ce traitement ?
   
   VOTRE RÉPONSE : [À RÉDIGER — 3 à 5 lignes]
''')


In [ ]:
# ============================================================
# EXERCICE 3 — Ajouter une transformation au journal de lineage
# ============================================================
# CONSIGNE : Ajoutez UNE transformation supplémentaire
# au journal de transformations de notre jeu de bibliothèque.
# Choisissez parmi :
#   Option A : Ajouter une colonne "domaine_disciplinaire"
#              en regroupant les filières en 4 grands domaines
#   Option B : Ajouter une colonne "semestre" calculée
#              depuis la date d'emprunt (sem. 1 = jan-juin, sem. 2 = juil-déc)

option_choisie = "[À COMPLÉTER : 'A' ou 'B']"

df_exercice = df_courant.copy()

if option_choisie == "A":
    # Option A : Regroupement en domaines disciplinaires
    mapping_domaines = {
        "Droit":                    "Sciences juridiques et politiques",
        "Sciences économiques":     "Sciences économiques et gestion",
        "Lettres modernes":         "Lettres, langues et arts",
        "Histoire":                 "Sciences humaines et sociales",
        "Géographie":               "Sciences humaines et sociales",
        "Médecine":                 "Sciences de la santé",
        "Agronomie":                "Sciences de la vie et de la terre",
        "Informatique":             "Sciences exactes et technologie",
        "Sciences de l'éducation":  "Sciences de l'éducation",
        "Sociologie":               "Sciences humaines et sociales",
        "Non renseignée":           "Non renseigné"
    }

    df_exercice["domaine_disciplinaire"] = df_exercice["filiere"].map(mapping_domaines)

    journal.enregistrer(
        df_avant     = df_courant,
        df_apres     = df_exercice,
        action       = "[À COMPLÉTER : décrivez l'action en 1 phrase]",
        raison       = "[À COMPLÉTER : pourquoi cette transformation est-elle utile ?]",
        outil        = "Python pandas — .map(dictionnaire)"
    )
    print("Option A appliquée.")
    print(df_exercice["domaine_disciplinaire"].value_counts())

elif option_choisie == "B":
    # Option B : Ajout du semestre
    df_exercice["semestre"] = pd.to_datetime(
        df_exercice["date_emprunt"]
    ).dt.month.apply(lambda m: "S1_Jan-Jun" if m <= 6 else "S2_Jul-Dec")

    journal.enregistrer(
        df_avant     = df_courant,
        df_apres     = df_exercice,
        action       = "[À COMPLÉTER : décrivez l'action en 1 phrase]",
        raison       = "[À COMPLÉTER : pourquoi ce champ est-il utile pour l'analyse ?]",
        outil        = "Python pandas — .dt.month + apply()"
    )
    print("Option B appliquée.")
    print(df_exercice["semestre"].value_counts())

else:
    print("⚠️ Choisissez 'A' ou 'B' dans option_choisie.")

# Sauvegarde du journal mis à jour
if option_choisie in ["A","B"]:
    journal.to_json("MIDA507_S02_journal_transformations_final.json")
    print("
✅ Journal de lineage mis à jour avec votre transformation.")


---
## PARTIE 7 · Export des Livrables


In [ ]:
# ============================================================
# CELLULE 7.1 — Export de tous les livrables S02
# ============================================================

livrables = [
    "MIDA507_S02_journal_cycle_vie.json",
    "MIDA507_S02_registre_checksums.json",
    "MIDA507_S02_DMP_BU_Ngaoundere.json",
    "MIDA507_S02_DMP_BU_Ngaoundere.md",
    "MIDA507_S02_journal_transformations.json",
    "MIDA507_S02_bibliotheque_ung_enrichie.csv",
]

print("LIVRABLES PRODUITS PAR LE NOTEBOOK S02")
print("=" * 55)
for fichier in livrables:
    if os.path.exists(fichier):
        taille = os.path.getsize(fichier)
        print(f"  ✅ {fichier:<45} ({taille:>6} octets)")
    else:
        print(f"  ⚠️  {fichier:<45} (non généré)")

try:
    from google.colab import files
    print("
Téléchargement Colab déclenché...")
    for f in livrables:
        if os.path.exists(f):
            files.download(f)
    print("✅ Tous les fichiers disponibles pour téléchargement.")
except ImportError:
    print("
⚠️  Exécution locale — fichiers sauvegardés dans le répertoire courant.")


---
## ✅ Bilan du Notebook S02 — Ce que vous avez accompli

| Compétence travaillée | Statut |
|---|---|
| Visualiser et documenter le cycle de vie DCC en 7 étapes | ✅ |
| Calculer un checksum SHA-256 pour la préservation | ✅ |
| Évaluer automatiquement les principes FAIR (14 critères) | ✅ |
| Comparer le niveau FAIR de portails africains réels | ✅ |
| Construire et exporter un DMP structuré (JSON + Markdown) | ✅ |
| Documenter le data lineage avec 4 transformations tracées | ✅ |
| Évaluer le FAIR de votre propre jeu (exercice 1) | 🟡 À compléter |
| Rédiger une section DMP pour un cas réel (exercice 2) | 🟡 À compléter |
| Ajouter une transformation au journal de lineage (exercice 3) | 🟡 À compléter |

---

## 🔗 Lien avec les sessions suivantes

| Session | Ce que vous utiliserez de S02 |
|---|---|
| **S03 — Open Data** | Le profil FAIR vous guidera dans l'évaluation des portails |
| **S04 — Métadonnées DCAT** | Le DMP fournit la structure des champs DCAT à renseigner |
| **S06 — Publication** | Le journal de lineage sera intégré dans la fiche DCAT publiée |
| **S07 — Éthique** | La section 4 du DMP (aspects légaux) est la base de l'anonymisation |

---

## 📎 Fichiers produits par ce notebook (6 livrables)

| Fichier | Contenu | Utilisé en |
|---|---|---|
| `journal_cycle_vie.json` | 7 étapes DCC documentées | S04, S06 |
| `registre_checksums.json` | Empreinte SHA-256 du jeu original | S06 préservation |
| `DMP_BU_Ngaoundere.json` | DMP structuré en JSON | S04, S06 |
| `DMP_BU_Ngaoundere.md` | DMP lisible en Markdown | Livrable professionnel |
| `journal_transformations.json` | Data lineage des 4 transformations | S04 fiche DCAT |
| `bibliotheque_ung_enrichie.csv` | Jeu enrichi (ARK + ISO 639 + pseudonymisation) | S04, S05, S06 |

---

*Notebook MIDA507 S02 · Master MIDA — Université de Douala · Cours Big Data, Open Data et Data Science pour l'IDA*
